<a href="https://colab.research.google.com/github/ultimatepin/card_recognizer/blob/main/notebooks/01_clip_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

colab is a cloud computer

! means a normal terminal command

## Clones the repo


In [ ]:
%cd /content

!rm -rf card_recognizer
!git clone https://github.com/ultimatepin/card_recognizer.git

%cd /content/card_recognizer

## Import libraries

In [ ]:
!pip -q install transformers pillow

In [ ]:
import os
import torch

from PIL import Image
from transformers import AutoProcessor, CLIPVisionModelWithProjection

In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch32"

# processor converts image to processable format
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# transformer and outputs embedding
model = CLIPVisionModelWithProjection.from_pretrained(MODEL_NAME)

# eval mode not train mode
model.eval()

## Turn image into embedding

In [ ]:
def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")

    # convert to pytorch tensor(pixel_values) to input model
    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    # not training the model, faster calc
    with torch.inference_mode():
        # pixel_values -> bunch of outputs
        output = model(**inputs)

    # embedding extraction
    embedding = output.image_embeds

    # Normalize the vector
    embedding = embedding / embedding.norm(dim=-1, keepdim=True)

    return embedding

`get_embedding()` returns a single embedding vector corresponding to `image_path`.

Cosine similarity of two images is calculated using dot product/norm, higher cos value means higher similarity, meaning the angle between vectors determines similarity.

## Create embeddings from card library

In [ ]:
card_folder = "cards"

card_embeddings = {}

for filename in os.listdir(card_folder):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        path = os.path.join(card_folder, filename)

        card_embeddings[filename] = get_embedding(path)

print(f"Loaded {len(card_embeddings)} cards.")

## Recognize `query.jpg`

In [ ]:
query_embedding = get_embedding("query.jpg")

results = []

for card_name, card_embedding in card_embeddings.items():

    # .item() returns a float instead of a torch
    similarity = torch.sum(
        query_embedding * card_embedding
    ).item()

    results.append((card_name, similarity))

results.sort(
    key=lambda x: x[1],
    reverse=True
)

for card_name, score in results:
    print(f"{card_name:25} {score:.4f}")

Working!

Principle: image -> CLIP(pixel_value -> embedding) -> cosine similarities -> winner

##